# E-Commerce Data Cleaning Project

## Overview
This notebook demonstrates a complete data cleaning pipeline for the UCI Online Retail dataset. The objective is to prepare the raw transactional data for analysis by identifying and resolving four key data quality issues:
1. Missing Values
2. Duplicate Records
3. Incorrect Data Types
4. Inconsistent Values

In [2]:
import pandas as pd
import numpy as np

## 1. Data Ingestion & Initial Exploration
First, we load the raw dataset and inspect its structure, dimensions, and numerical distributions to understand our baseline.

In [2]:
df = pd.read_csv('online_retail.csv')
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [3]:
print(f"total rows {df.shape[0]:,}")
print(f"total columns {df.shape[1]}")

total rows 541,909
total columns 8


In [4]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0       InvoiceNo  541909 non-null  object 
 1   StockCode      541909 non-null  object 
 2   Description    540455 non-null  object 
 3   Quantity       541909 non-null  int64  
 4   InvoiceDate    541909 non-null  object 
 5   UnitPrice      541909 non-null  float64
 6   CustomerID     406829 non-null  float64
 7   Country        541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## 2. Handling Missing Values
In transactional datasets, missing essential identifiers (like Customer ID) make the record unusable for customer-centric analysis. We will measure the percentage of missing data and drop the unusable rows.

In [5]:
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct

    InvoiceNo     0.000000
StockCode         0.000000
Description       0.268311
Quantity          0.000000
InvoiceDate       0.000000
UnitPrice         0.000000
CustomerID       24.926694
Country           0.000000
dtype: float64

In [6]:

initial_rows = len(df)
initial_rows


541909

In [7]:
df = df.dropna(subset=['CustomerID']).copy()
df = df.dropna(subset=['Description']).copy()

In [8]:
print(df.isnull().sum())
print("length of the dataframe:", len(df))

    InvoiceNo    0
StockCode        0
Description      0
Quantity         0
InvoiceDate      0
UnitPrice        0
CustomerID       0
Country          0
dtype: int64
length of the dataframe: 406829


## 3. Removing Duplicate Records
System errors or user retries can create identical duplicate rows. We will identify and remove any exact row-level matches.

In [9]:
duplicate_count = df.duplicated().sum()
print("duplicated rows :", duplicate_count)
df = df.drop_duplicates(keep='first').copy()

duplicated rows : 5225


In [10]:
df.duplicated().sum()

np.int64(0)

## 4. Correcting Data Types
To optimize memory and allow for time-series analysis, we must cast columns to their appropriate data types.
* `InvoiceDate` is converted to a DateTime object.
* `CustomerID` is converted to a string (since it is an identifier, not a mathematical integer).
* `Country` is converted to a categorical type to save memory.

In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [12]:
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

In [13]:
df['Country'] = df['Country'].astype('category')

In [14]:
print(df.dtypes)

    InvoiceNo            object
StockCode                object
Description              object
Quantity                  int64
InvoiceDate      datetime64[ns]
UnitPrice               float64
CustomerID               object
Country                category
dtype: object


## 5. Fixing Inconsistent Values
Real-world data often contains logical inconsistencies. We will:
1. Filter out cancelled orders (negative/zero `Quantity`) and accounting errors (negative/zero `UnitPrice`).
2. Standardize text strings in the `Description` column to prevent duplicate categories caused by varying capitalization or trailing spaces.

In [15]:
print("rows with negative or zero unit price:", df[df['UnitPrice'] <= 0].count())
print("rows with negative or zero quantity:", df[df['Quantity'] <= 0].count())
print("length of the dataframe before filtering:", len(df))


rows with negative or zero unit price:     InvoiceNo    40
StockCode        40
Description      40
Quantity         40
InvoiceDate      40
UnitPrice        40
CustomerID       40
Country          40
dtype: int64
rows with negative or zero quantity:     InvoiceNo    8872
StockCode        8872
Description      8872
Quantity         8872
InvoiceDate      8872
UnitPrice        8872
CustomerID       8872
Country          8872
dtype: int64
length of the dataframe before filtering: 401604


In [16]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

In [17]:
len(df)

392692

In [18]:
df['Description'] = df['Description'].str.strip().str.upper()

In [19]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


## 6. Export Cleaned Dataset
With all missing values, duplicates, bad data types, and inconsistencies resolved, the dataset is exported for downstream analysis.

In [20]:
df.to_csv('cleaned_online_retail_data_v1.csv', index=False)

In [21]:
df.columns

Index(['    InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [3]:
eda = pd.read_csv('cleaned_online_retail_data_v1.csv')

In [7]:
eda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    392692 non-null  int64  
 1   StockCode    392692 non-null  object 
 2   Description  392692 non-null  object 
 3   Quantity     392692 non-null  int64  
 4   InvoiceDate  392692 non-null  object 
 5   UnitPrice    392692 non-null  float64
 6   CustomerID   392692 non-null  int64  
 7   Country      392692 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 24.0+ MB
